# 🧠 ADHDLab: Colab L4 × Qwen 2.5 (14B) ADHD特化型自己理解＆支援AIツール

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/9z2fqjckjv-bot/ColabTool/blob/main/ADHDLab/ADHDLab_Colab.ipynb)

Google Colab の強力な **NVIDIA L4 GPU (24GB VRAM)** と **Qwen 2.5 (14B)** を組み合わせ、Google One AI Ultra（20TB）のストレージ連携を活かした **ADHD特化型自己理解・タスク管理・メンタルケア支援ツール** です。

---

### 🌟 ADHDLabの主要機能
- **🧠 ADHD特化型 LLM (Qwen 2.5 14B)**: L4 GPU向け4bit量子化（BitsAndBytes）により、VRAM約9GBで超高速＆高精度に動作。先延ばし防止のマイクロステップ法、視覚的タスク分解、ドーパミン報酬系ハックを自動適用。
- **📚 「ソース一覧.xlsx」完全準拠ナレッジベース**: 医療機関・公的機関・SNS・Blog等の最新エビデンスを自動収集・分類・優先度ソート（sampleシート規則準拠）。RAGにより信頼性の高い根拠を回答に自動引用。
- **✨ Geminiライクな洗練されたモダンWeb UI**: ブラウザから利用できる美しいUI。チャット履歴、メディア添付、ナレッジベース探索、ワンクリック自動収集を統合。
- **💾 トリプル永続化同期**: Google Drive ()、GitHub (自動プッシュ)、XServer (SSH ) へチャットログや収集データを自動保管。
- **💰 追加費用0円**: Google ColabとオープンソースLLM、無料Web検索を活用し、完全無料（0円）で運用可能。

---
## 📍 ステップ 0: ハードウェア診断（L4 GPU & VRAM確認）
Google Colab のGPU環境（L4 GPU 24GB VRAM 推奨）を診断します。

In [ ]:
#@title 📍 ステップ 0: GPU診断＆L4 GPU判定
import torch
from IPython.display import display, HTML

has_gpu = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if has_gpu else "GPU未検出 (CPU)"
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if has_gpu else 0
is_l4 = "L4" in gpu_name

status_color = "#10b981" if is_l4 else ("#3b82f6" if has_gpu and vram_gb >= 15 else "#f59e0b")
status_title = f"🎉 {gpu_name} (VRAM: {vram_gb:.1f} GB) で接続中！" if has_gpu else "⚠️ GPUが割り当てられていません"
status_msg = "Qwen 2.5 (14B) 4bit量子化モデルを最高パフォーマンスで実行できる理想的な環境です。" if is_l4 else (
    f"検出されたGPU: {gpu_name} ({vram_gb:.1f}GB)。実行可能です。" if has_gpu else "上部メニューの [ランタイム] ➔ [ランタイムのタイプを変更] から [L4 GPU] を選択してください。"
)

display(HTML(f"""
<div style="background: #0f172a; padding: 20px; border-radius: 12px; border-left: 6px solid {status_color}; font-family: sans-serif;">
  <h3 style="color: {status_color}; margin: 0 0 8px 0; font-size: 17px;">{status_title}</h3>
  <p style="color: #e2e8f0; font-size: 13.5px; margin: 0;">{status_msg}</p>
</div>
"""))


---
## 💾 ステップ 1: Google Drive マウント（永続化ディレクトリ設定）
Google One AI Ultra（20TB）等のGoogle Driveをマウントし、チャットログや収集データ、生成メディアを自動保管します。
※世界へ公開された場合も、実行したユーザー自身のGoogle Driveに安全に保管されます。

In [ ]:
#@title 💾 ステップ 1: Google Drive マウント
import os
from google.colab import drive

print("📁 Google Driveをマウントしています...")
drive.mount("/content/drive")

drive_adhd_dir = "/content/drive/MyDrive/ADHDLab"
os.makedirs(drive_adhd_dir, exist_ok=True)
print(f"✅ Google Drive 永続化ディレクトリを準備しました: {drive_adhd_dir}")


---
## 📦 ステップ 2: リポジトリの準備 & 依存ライブラリの自動インストール
最新のコードを取得し、高速推論に必要なライブラリ群をインストールします。

In [ ]:
#@title 📦 ステップ 2: 依存ライブラリのインストール
import os, sys, subprocess

# 作業ディレクトリの設定
repo_dir = "/content/ColabTool/ADHDLab"
if not os.path.exists("/content/ColabTool"): 
    print("📥 リポジトリをクローン中...")
    subprocess.run(["git", "clone", "https://github.com/9z2fqjckjv-bot/ColabTool.git", "/content/ColabTool"], check=True)
else:
    print("🔄 リポジトリを最新に更新中...")
    subprocess.run(["git", "pull"], cwd="/content/ColabTool")

os.chdir(repo_dir)
sys.path.insert(0, repo_dir)

print("⚡ 高速推論 & WebUI 依存パッケージをインストール中...")
!pip install -q transformers accelerate bitsandbytes sentencepiece openpyxl duckduckgo-search fastapi uvicorn pydantic pyngrok nest_asyncio

print("✅ 依存パッケージの準備が完了しました！")


---
## 🚀 ステップ 3: Qwen 2.5 (14B) モデルのロード
L4 GPU（24GB VRAM）のパワーを活かし、**Qwen/Qwen2.5-14B-Instruct** を4bit量子化（NF4）でロードします。
※VRAM消費は約9.5GBに抑えられ、余裕を持った超高速推論が可能です。

In [ ]:
#@title 🚀 ステップ 3: Qwen 2.5 (14B) モデルのロード
from core.llm_engine import LLMEngine
from web.app import llm_engine

selected_model = "Qwen/Qwen2.5-14B-Instruct" #@param ["Qwen/Qwen2.5-14B-Instruct", "Qwen/Qwen2.5-7B-Instruct"]

print(f"🔥 {selected_model} (4bit 量子化 on L4 GPU) をVRAMにロード中...")
llm_engine.load_model(selected_model)
print("✅ モデルの準備が整いました！")


---
## 🌐 ステップ 4: 【ワンクリック起動】ADHDLab Web UI（Geminiスタイル）
FastAPIサーバーとトンネルを起動し、ブラウザ（MacBook / PC / スマホ）からアクセスできる公開URLを発行します。

> [!TIP]
> セル実行後、下部に表示される **「🚀 ADHDLab チャット画面を開く」** ボタンをクリックしてください！

In [ ]:
#@title 🌐 ステップ 4: Web UI サーバー起動 & 公開URL発行
import os, time, subprocess, threading
from IPython.display import display, HTML
import nest_asyncio
nest_asyncio.apply()

# 1. FastAPI サーバーをバックグラウンド起動
def run_server():
    import uvicorn
    from web.app import app
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)

# 2. Cloudflare トンネルによるパブリックURL発行
print("🌐 セキュアなパブリックURLを発行中 (Cloudflare Tunnel)...
")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in tunnel_proc.stdout:
    if "trycloudflare.com" in line:
        for word in line.split():
            if "trycloudflare.com" in word:
                public_url = word.strip()
                if not public_url.startswith("http"):
                    public_url = "https://" + public_url
                break
        if public_url:
            break

if not public_url:
    public_url = "http://127.0.0.1:8000"

display(HTML(f"""
<div style="background: linear-gradient(135deg, #1e3a8a 0%, #3b82f6 100%); padding: 26px; border-radius: 16px; color: white; font-family: sans-serif; box-shadow: 0 10px 25px rgba(37, 99, 235, 0.25); text-align: center;">
  <h2 style="margin: 0 0 10px 0; font-size: 22px;">✨ ADHDLab サーバーが正常に起動しました！</h2>
  <p style="font-size: 14px; opacity: 0.9; margin-bottom: 20px;">下のボタンをクリックすると、GeminiスタイルのWeb UIが新しいタブで開きます。</p>
  <a href="{public_url}" target="_blank" style="display: inline-block; background: #ffffff; color: #1e3a8a; font-weight: 700; font-size: 16px; padding: 14px 32px; border-radius: 30px; text-decoration: none; box-shadow: 0 4px 14px rgba(0,0,0,0.15); transition: transform 0.15s ease;">
    🚀 今すぐ ADHDLab を開く
  </a>
  <div style="margin-top: 15px; font-size: 12px; opacity: 0.75;">
    URL: <a href="{public_url}" target="_blank" style="color: #bfdbfe;">{public_url}</a>
  </div>
</div>
"""))


---
## 💾 ステップ 5: データの包括的同期・バックアップ（随時・セッション終了時）
Google Drive、GitHub、XServer（SSH ）へ、チャット履歴・収集データ・Excelスプレッドシートを一括同期・保存します。

In [ ]:
#@title 💾 ステップ 5: Google Drive / GitHub / XServer 包括的同期
import os
from core.storage_sync import StorageSyncManager

# GitHub Personal Access Token (Colab Secretsの "GITHUB_TOKEN" または下の入力欄)
github_token_input = "" #@param {type:"string"}
if github_token_input:
    os.environ["GITHUB_TOKEN"] = github_token_input.strip()

sync_mgr = StorageSyncManager()
print("🔄 全保管先への一括バックアップを開始します...")
results = sync_mgr.full_sync(commit_msg="ADHDLab Colab Session Backup")

print("
--- [同期結果] ---")
print(f"1. Google Drive: {results['google_drive']['message']}")
print(f"2. GitHub      : {results['github']['message']}")
print(f"3. XServer     : {results['xserver']['message']}")
print("------------------")
print("🎉 すべてのデータが安全に永続化されました！")
